In [0]:
pip install asyncpraw

In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
import asyncpraw
import pandas as pd
from delta.tables import DeltaTable

delta_table_name = "default.reddit_content_bronze"
BATCH_SIZE = 100
SEARCH_TERM = "databricks"

async def main():
    reddit = asyncpraw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent
    )

    posts = []
    after = None
    while True:
        params = {"limit": BATCH_SIZE}
        if after:
            params["after"] = after

        batch = []
        subreddit = await reddit.subreddit("all")
        async for submission in subreddit.search(
            query=SEARCH_TERM,
            sort="new",
            syntax="lucene",
            time_filter="month",
            params=params
        ):
            await submission.load()
            await submission.comments.replace_more(limit=None)
            all_comments = await submission.comments.list()
            comments_data = [
                {
                    "author": comment.author.name if comment.author else None,
                    "score": comment.score,
                    "body": comment.body,
                    "timestamp": comment.created_utc
                }
                for comment in all_comments if comment.body
            ]
            batch.append({
                "title": submission.title,
                "url": submission.url,
                "score": submission.score,
                "num_comments": submission.num_comments,
                "timestamp": submission.created_utc,
                "subreddit": submission.subreddit.display_name,
                "comments": comments_data
            })
            last_fullname = submission.fullname

        if not batch:
            break

        # Write batch to Delta
        pdf = pd.DataFrame(batch)
        df = spark.createDataFrame(pdf)
        upsert_to_delta(df, delta_table_name)

        after = last_fullname

    await reddit.close()

def upsert_to_delta(df, table_name):
    df_deduped = df.dropDuplicates(["url"])
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        (
            delta_table.alias("t")
            .merge(
                df_deduped.alias("s"),
                "t.url = s.url"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_deduped.write.format("delta").mode("overwrite").saveAsTable(table_name)

await main()

In [0]:
%sql
SELECT * FROM  reddit_content_bronze